# 04a — EXP_01: RAGAS Evaluation (Claude Sonnet 4.6 judge)

Scores `results/exp_01_base_llm__golden_234/predictions.jsonl` against the 234-row golden subset using the locked judge.

**Three families separation upheld**: generator = LLaMA · constructor = `gpt-4o` · judge = `claude-sonnet-4-6` (locked 2026-05-06, upgraded from `claude-3-5-sonnet-20241022` — same per-token pricing, materially better structured-output adherence + sub-statement claim verification).

## What gets scored — and what's deliberately skipped

EXP_01 has **no retrieved context**. RAGAS's context-dependent metrics (Faithfulness, Context Precision, Context Recall) are undefined when `retrieved_contexts = []`, so we report them as `null` in `summary.json` rather than fabricating a 0/1 value. Methodology framing is in [`docs/output_notes/04a_exp01_output.md` §3](../docs/output_notes/04a_exp01_output.md) and [`docs/results_schema.md` §2.3](../docs/results_schema.md).

| Metric | Run for EXP_01? | Why |
|---|---|---|
| Answer Relevancy | ✓ | Question + answer only; doesn't need context |
| Answer Correctness | ✓ | Answer + reference only; doesn't need context |
| Faithfulness | skip | Undefined without retrieved evidence |
| Context Precision | skip | Same |
| Context Recall | skip | Same |

EXP_02–EXP_05 (which retrieve real chunks) will run all five.

## Two gated stages

| Stage | Surface | Wall time | Cost (Claude Sonnet 4.6) |
|---|---|---|---|
| **A** Smoke | 10 rows | ~30 s | **~$0.50** |
| **B** Full | 234 rows | ~5–10 min | **~$8–12** |

> **Cost recalibrated 2026-05-06**: the original notebook estimate of $0.30 / $3–5 was ~3× optimistic — Answer Correctness alone makes 2–3 calls per row (atomise answer + reference + classify TP/FP/FN). Real per-row cost for the 2 EXP_01 metrics ≈ $0.04–0.05.

Stage B only fires when `RUN_FULL = True`. Once `ragas_scores.csv` exists, the runner skips judging on re-runs (file-level resumability).

## 1. Setup

In [1]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.eval.ragas_eval import score_predictions

# Sanity: keys must be present
for k in ["ANTHROPIC_API_KEY"]:
    print(f"{k}: {'\u2713 present' if os.environ.get(k) else '\u2717 MISSING — add to .env at repo root and reload'}")

print("Repo root:", REPO_ROOT)

ANTHROPIC_API_KEY: ✓ present
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project


## 2. Configuration

In [2]:
PRED_DIR = REPO_ROOT / "results" / "exp_01_base_llm__golden_234"
GOLDEN_PATH = REPO_ROOT / "data" / "processed" / "golden_ragas_300.jsonl"
CHUNKS_PATH = REPO_ROOT / "data" / "processed" / "chunks.parquet"

RUN_SMOKE = False
SMOKE_N = 10
RUN_FULL = False  # flip to True only after the smoke looks clean
RUN_RESCORE_NANS = True  # Stage C — flip to True to re-judge rows that came back NaN; merges into existing CSV
OVERWRITE_SCORES = False  # True forces a fresh judge run, ignoring existing ragas_scores.csv (mutually exclusive with RUN_RESCORE_NANS)

print("Predictions:", PRED_DIR)
print("Golden     :", GOLDEN_PATH)
print("Chunks     :", CHUNKS_PATH)

Predictions: /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_01_base_llm__golden_234
Golden     : /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/golden_ragas_300.jsonl
Chunks     : /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/chunks.parquet


## 3. Stage A — Smoke (10 rows · ~30 s · ~$0.30)

Runs Answer Relevancy + Answer Correctness on the first 10 joined rows. Acceptance gates printed below the cell.

> **Note on scores file**: smoke writes to a separate scratch path (`ragas_scores_smoke.csv`) so the full-234 file isn't polluted with a partial 10-row CSV.

In [3]:
import pandas as pd
import shutil

if RUN_SMOKE:
    smoke_dir = PRED_DIR.parent / "exp_01_base_llm__golden_234_ragas_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    # Mirror the three input files into the smoke dir so summary.json updates land in the scratch dir, not the canonical one.
    for fname in ("predictions.jsonl", "retrieval.jsonl", "summary.json"):
        shutil.copy2(PRED_DIR / fname, smoke_dir / fname)

    smoke_summary = score_predictions(
        predictions_dir=smoke_dir,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=SMOKE_N,
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage A smoke summary (10 rows) ===")
    for k, v in smoke_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")

    smoke_scores = pd.read_csv(smoke_dir / "ragas_scores.csv")
    print("\n=== Per-row scores (smoke 10) ===")
    cols_to_show = [c for c in smoke_scores.columns if c in ("question_id", "_is_correct", "answer_relevancy", "answer_correctness")]
    print(smoke_scores[cols_to_show].to_string(index=False))
else:
    print("RUN_SMOKE = False — skipped.")

[ragas] joined 10 rows | metrics = ['answer_correctness', 'answer_relevancy']


/Users/rajak/Workstation/Projects/myGitHub/thesis-project/src/eval/ragas_eval.py:229: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=embed_model))


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

[ragas] judge wall-time 160.9 s for 10 rows × 2 metrics
[ragas] wrote /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_01_base_llm__golden_234_ragas_smoke/ragas_scores.csv

=== Stage A smoke summary (10 rows) ===
  RAGAS_Faithfulness             : None
  RAGAS_Hallucination_Rate       : None
  RAGAS_Answer_Relevance         : 0.6006195019693744
  RAGAS_Context_Precision        : None
  RAGAS_Context_Recall           : None
  Answer_Correctness             : 0.7959772106805426
  ragas_metrics_run              : ['answer_correctness', 'answer_relevancy']
  ragas_n_scored                 : 10
  ragas_judge                    : claude-sonnet-4-6
  ragas_timestamp_utc            : 2026-05-06T15:10:07Z

=== Per-row scores (smoke 10) ===
 answer_correctness  answer_relevancy question_id  _is_correct
           0.345729          0.591925  golden_000        False
           0.931818          0.561127  golden_001         True
           0.942308          0.723649  golden_002

**Acceptance gates (Stage A)** — flip `RUN_FULL = True` only when all four pass:

1. **No errors thrown** by the `evaluate()` call (RAGAS prints a progress bar then a result; if it raises a Pydantic validation error or an Anthropic API error, fix before scaling).
2. **Both metric columns exist** in the scores CSV: `answer_relevancy`, `answer_correctness`.
3. **No NaN-flood** — at most 1–2 NaNs per column on a 10-row pilot is normal (the judge sometimes refuses on edge cases). > 30% NaN means a wiring problem.
4. **Sane score distribution** — Answer Correctness should correlate with `_is_correct`: rows where `_is_correct=True` should average higher than `_is_correct=False`. A flat distribution suggests the prose-conversion or reference field is mismatched.

## 4. Stage B — Full 234 (~5–10 min · ~$3–5)

Writes to the canonical `results/exp_01_base_llm__golden_234/` directory. Updates `summary.json` in place. `ragas_scores.csv` is the per-row artifact for downstream LIME/SHAP and confidence-signal extraction (Phases 6–7).

In [5]:
if RUN_FULL:
    full_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=None,  # full 234
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage B full summary (234 rows) ===")
    for k, v in full_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")) or k == "Acuuracy":
            print(f"  {k:30s} : {v}")
else:
    print("RUN_FULL = False — set to True after Stage A passes the four gates.")

[ragas] joined 234 rows | metrics = ['answer_correctness', 'answer_relevancy']


Evaluating:   0%|          | 0/468 [00:00<?, ?it/s]

Exception raised in Job[1]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[46]: TimeoutError()
Exception raised in Job[58]: TimeoutError()
Exception raised in Job[56]: TimeoutError()
Exception raised in Job[82]: TimeoutError()
Exception raised in Job[100]: TimeoutError()
Exception raised in Job[102]: TimeoutError()
Exception raised in Job[103]: TimeoutError()
Exception raised in Job[104]: TimeoutError()
Exception raised in Job[105]: TimeoutError()
Exception raised in Job[106]: TimeoutError()
Exception raised in Job[107]: TimeoutError()
Exception raised in Job[108]: TimeoutError()
Exception raised in Job[109]: TimeoutError()
Exception raised in Job[110]: TimeoutError()
Exception raised in Job[112]: TimeoutError()
Exception raised in Job[113]: TimeoutError()
Exception raised in Job[114]: TimeoutError()
Exception raised in Job[115]: TimeoutError()
Exception raised in Job[116]: TimeoutError()
Exception raised in Job[118]: TimeoutError()
Exception raised 

[ragas] judge wall-time 2846.3 s for 234 rows × 2 metrics
[ragas] wrote /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_01_base_llm__golden_234/ragas_scores.csv

=== Stage B full summary (234 rows) ===
  Acuuracy                       : 0.9017094017094017
  RAGAS_Faithfulness             : None
  RAGAS_Hallucination_Rate       : None
  RAGAS_Answer_Relevance         : 0.5976971700606776
  RAGAS_Context_Precision        : None
  RAGAS_Context_Recall           : None
  Answer_Correctness             : 0.8738074347344713
  ragas_metrics_run              : ['answer_correctness', 'answer_relevancy']
  ragas_n_scored                 : 234
  ragas_judge                    : claude-sonnet-4-6
  ragas_timestamp_utc            : 2026-05-06T16:02:11Z


## 5. Stage C — NaN rescore (re-judge transient-failure rows)

The first pass through Stage B can leave NaN cells in `ragas_scores.csv` from transient Anthropic API failures (rate-limit throttles absorbed by `raise_exceptions=False`). This stage re-judges only those rows and merges new scores into the existing CSV.

**The runner now uses `RunConfig(max_workers=4, max_retries=10, max_wait=120)`** which should keep the on-first-pass NaN rate low. Stage C is the recovery path for any NaN that still slip through.

**Cost** — only the NaN rows pay. For EXP_01, the 2026-05-06 first run left 97 / 234 NaN on AnswerCorrectness + 60 / 234 NaN on AnswerRelevancy = ~121 unique rows × ~$0.04/row ≈ **~$5** to complete. Already-good cells are *not* overwritten.

In [ ]:
if RUN_RESCORE_NANS:
    rescored_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        rescore_nans=True,
    )
    print("\n=== After NaN rescore ===")
    for k, v in rescored_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")

    # Show how many NaN remain after the rescore (should be 0 or near-zero if all transient)
    df = pd.read_csv(PRED_DIR / "ragas_scores.csv")
    for col in ("answer_relevancy", "answer_correctness"):
        if col in df.columns:
            n_nan = pd.to_numeric(df[col], errors="coerce").isna().sum()
            print(f"  {col}: {n_nan} NaN remaining of {len(df)}")
else:
    print("RUN_RESCORE_NANS = False — set to True after Stage B if NaN rate is non-trivial.")

[ragas] joined 234 rows | metrics = ['answer_correctness', 'answer_relevancy']
[ragas] rescoring 121 NaN rows (121 unique question_ids)


/Users/rajak/Workstation/Projects/myGitHub/thesis-project/src/eval/ragas_eval.py:240: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=embed_model))


[ragas] RunConfig: max_workers=4, max_retries=10, max_wait=120s, timeout=180s


Evaluating:   0%|          | 0/242 [00:00<?, ?it/s]

KeyboardInterrupt: 

Exception raised in Job[38]: AssertionError(AnswerSimilarity must be set)
Exception raised in Job[40]: AssertionError(llm is not set)
Exception raised in Job[44]: AssertionError(LLM must be set)
Exception raised in Job[45]: AssertionError(LLM is not set)
Exception raised in Job[46]: AssertionError(LLM must be set)
Exception raised in Job[47]: AssertionError(LLM is not set)
Exception raised in Job[48]: AssertionError(LLM must be set)
Exception raised in Job[49]: AssertionError(LLM is not set)
Exception raised in Job[50]: AssertionError(LLM must be set)
Exception raised in Job[51]: AssertionError(LLM is not set)
Exception raised in Job[52]: AssertionError(LLM must be set)
Exception raised in Job[53]: AssertionError(LLM is not set)
Exception raised in Job[54]: AssertionError(LLM must be set)
Exception raised in Job[55]: AssertionError(LLM is not set)
Exception raised in Job[56]: AssertionError(LLM must be set)
Exception raised in Job[57]: AssertionError(LLM is not set)
Exception raised in

## 6. Inspect — Answer Correctness vs Exact Match correlation

Sanity check: rows the LLM got *right* (`_is_correct=True`) should average higher Answer Correctness than rows it got *wrong*. If the gap isn't visible, something is off in the prose-conversion or reference field.

In [6]:
scores_path = PRED_DIR / "ragas_scores.csv"
if scores_path.exists():
    df = pd.read_csv(scores_path)
    print(f"\nrows: {len(df)}")
    for col in ("answer_relevancy", "answer_correctness"):
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        print(f"\n  {col} distribution:")
        print(f"    mean={s.mean():.4f}  std={s.std():.4f}  NaN={s.isna().sum()}")
        print("    by _is_correct:")
        print(df.groupby("_is_correct")[col].agg(["mean", "count"]).round(4))
else:
    print("No ragas_scores.csv yet — run Stage B first.")


rows: 234

  answer_relevancy distribution:
    mean=0.5977  std=0.0768  NaN=60
    by _is_correct:
               mean  count
_is_correct               
False        0.5854     19
True         0.5992    155

  answer_correctness distribution:
    mean=0.8738  std=0.2194  NaN=97
    by _is_correct:
               mean  count
_is_correct               
False        0.3139     13
True         0.9325    124


---

**Next.** With EXP_01's RAGAS row complete, the next session builds `src/retrieval/naive.py` and the EXP_02 notebook (`04b_exp02_naive_rag.ipynb`). EXP_02 reuses this same RAGAS notebook structure with all five metrics.